# Retrieve NASA SWOT data

*Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access*, Session 3, Tuesday, November 10, 2026

**Learning goal:** Retrieve NASA SWOT data for a location of interest, describe what comes back using the course's shared concepts, and choose an access pattern that scales with your question.

You will retrieve river data from the NASA Surface Water and Ocean Topography (SWOT) mission with Python: find the files that cover a river, download them, and pull a water surface elevation time series for one river reach. The example is the 2025 spring flood of the Ohio River at Louisville, Kentucky.

The slides explain the ideas, and this notebook is for running the code. Run the cells in order.

## Before you start

**Setup.** Follow the setup instructions in the HydroShare README, then choose the **Python (open-water-data)** kernel. Create your Earthdata Login before the session.

**Downloads and time.** The main sections download about 230 MB of SWOT files into `data/`, in about 2 minutes. "Now you try it" adds one RiverSP file and two Raster scenes. On Windows, keep the folder path short (for example `C:\fwdc\`): SWOT file names are about 100 characters, and Windows refuses paths over 260, with `FileNotFoundError: [WinError 3]`.

**Results can change.** Search results and values can change when NASA reprocesses the data (a new CRID or version), and recent overpasses are added as they're processed.

**Live coding 1: find SWOT data.** Run from here to the end of *Find RiverSP granules over the Ohio at Louisville*, then go back to the slides.

## Credentials

**NASA Earthdata Login.** Downloading SWOT files needs a free account from [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov). `earthaccess.login()` looks for your username and password in the `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` environment variables, then in a `~/.netrc` file, and otherwise asks you for them. In a hosted Jupyter environment, the simplest route is to let it ask: run the cell and type your details into the boxes it shows. To avoid typing them every session, run `earthaccess.login(persist=True)` once, which saves them to `~/.netrc` in your home folder. Never type your password into a code cell: it would be saved in the notebook. See the [`earthaccess` authentication how-to](https://earthaccess.readthedocs.io/en/latest/user/howto/authenticate/).

You should see `Logged in to NASA Earthdata: True`.

In [ ]:
import earthaccess

# Uses EARTHDATA_USERNAME/EARTHDATA_PASSWORD or ~/.netrc if set; otherwise asks you to type them
auth = earthaccess.login()
print("Logged in to NASA Earthdata:", auth.authenticated)

## Define your query

Put everything that defines your query in one place. Every later cell reads these variables, so changing the river or the dates starts here, and saving it records exactly what you asked for. For your own river, change the name, the point and the dates. A few later cells also name one overpass (such as pass `175`): change those to an overpass of your river.

The dates are full UTC timestamps (`Z` means UTC) because both `earthaccess` and `hydrocron` accept that form. You won't know `reach_id` for a new river yet: the RiverSP download below shows how to find it.

In [ ]:
# The lesson's query, in one place. Change these to use another river.
site_name = "Ohio River at Louisville, KY"
site_lon, site_lat = -85.799, 38.280          # a point on the river (decimal degrees, WGS84)
site_bbox = (site_lon - 0.02, site_lat - 0.02, site_lon + 0.02, site_lat + 0.02)  # (west, south, east, north), ~3.5 x 4.4 km
start, end = "2025-03-15T00:00:00Z", "2025-05-15T23:59:59Z"   # UTC; the April 2025 flood falls inside
reach_id = "74267300251"                      # SWORD reach at the point (found below, or look it up in SWORD Explorer)
data_dir = "data/swot/raw"                    # downloads go here and are never edited

## Programmatic data discovery

### Find datasets

If you don't yet know which dataset you need, search the catalog broadly with `search_datasets` (see the [`earthaccess` API docs](https://earthaccess.readthedocs.io/en/latest/api/) for every option). A keyword alone matches more than 1,500 datasets, so this cell adds your area and dates, and keeps only cloud-hosted ones.

In [ ]:
# Datasets matching "river" that are in the cloud and cover the site during the window
river_datasets = earthaccess.search_datasets(
    keyword="river",
    cloud_hosted=True,
    bounding_box=site_bbox,
    temporal=(start, end),
)
short_names = [d["umm"]["ShortName"] for d in river_datasets]
print(len(short_names), "datasets;", "SWOT ones:")
sorted(name for name in short_names if name.startswith("SWOT"))

You should see about 47 datasets and a list of SWOT short names. Each result is a *collection*, and its **short name** is what you pass to the other `earthaccess` functions.

- **Version.** Names ending in `_D` are **Version D**, the current release. Names ending in `_2.0` are the older Version C. Use Version D.
- **Sub-collections.** `SWOT_L2_HR_RiverSP_D` holds both reach and node files; `_reach_D` and `_node_D` are sub-collections with one kind each.
- **Raster isn't here**: it doesn't match this keyword search. Searching for `keyword="SWOT"` finds it.

### A river with no SWOT reaches: the Mississippi headwaters

`search_data` finds the granules inside one collection. A quick lesson first: **a granule in your results is not a guaranteed observation of your site.** Each RiverSP granule covers a whole pass across a continent, so its footprint can "intersect" a small box even when SWOT measured no river there. Here is a box at the Mississippi River headwaters in Minnesota, a small stream at that point.

In [ ]:
import geopandas as gpd
from shapely.geometry import box

headwaters_box = (-95.26, 47.17, -95.15, 47.25)  # (west, south, east, north) at Lake Itasca, MN
headwaters = earthaccess.search_data(
    short_name="SWOT_L2_HR_RiverSP_D",
    granule_name="*Reach*",          # reach files only (this collection also holds node files)
    bounding_box=headwaters_box,
    temporal=("2026-06-01", "2026-06-30T23:59:59"),
)
print(len(headwaters), "reach granules found")
files = earthaccess.download(headwaters[:1], local_path=data_dir)
reaches = gpd.read_file(files[0])  # geopandas reads the zipped shapefile directly
print(len(reaches), "reaches in the first granule;", int(reaches.intersects(box(*headwaters_box)).sum()), "inside the box")

You should see 42 reach granules, and 650 reaches in the first granule with 0 inside the box. The same is true for all 42 June 2026 granules. The problem is not SWOT's orbit: SWORD, the river network that RiverSP reports on, has no reaches in this box, because the Mississippi here is far narrower than the rivers SWOT is built to measure. Always open one file and check before you download many.

### Find Raster granules over the Ohio at Louisville

Now search the 100 m Raster product in the small box around the example point.

In [ ]:
louisville_raster = earthaccess.search_data(
    short_name="SWOT_L2_HR_Raster_100m_D",
    bounding_box=site_bbox,
    temporal=(start, end),
)
print(len(louisville_raster), "granules")
print(sorted({g["umm"]["GranuleUR"].split("_")[5] for g in louisville_raster}))  # the UTM zone + band in each name

You should see 44 granules, with UTM zones `UTM01C`, `UTM01W`, `UTM16S`, `UTM60C`, `UTM60V` and `UTM60W`.

That is far more than SWOT could have seen over one point in two months. The UTM zone (Universal Transverse Mercator system) in each granule name gives the problem away. Louisville is in UTM zone **16**, but most results are in zones 01 and 60, on either side of the 180° meridian. For the Raster data products, the metadata footprints of those granules wrap around the whole globe, so they falsely "intersect" almost any box. Downloading them all would waste gigabytes on scenes from the other side of the planet. This is a known issue, and PO.DAAC is working to resolve it.

In the meantime, a more reliable check is each granule's footprint *polygon* (`GPolygons` in its metadata). The false matches have no polygon, only a global rectangle. The first function below keeps the granules whose footprint polygon contains your point.

The second function handles duplicates. A granule name ends in two codes: the **CRID** (composite release identifier, for example `PGD0`) and a **product counter** (`01`, `02`, …). `PGD0` is reprocessed Version D, `PID0` is forward-processed Version D, and codes ending in `C0`–`C2` are older Version C. Where *both* exist, use `PGD0`, and use the highest product counter. The same overpass can turn up under more than one CRID, so the function keeps one granule per overpass.

In [ ]:
from shapely.geometry import Point, Polygon

def footprint_covers(granule, lon, lat):
    """True if one of the granule's footprint polygons contains the point (lon, lat)."""
    geometry = granule["umm"]["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]
    for polygon in geometry.get("GPolygons", []):
        points = [(p["Longitude"], p["Latitude"]) for p in polygon["Boundary"]["Points"]]
        if Polygon(points).contains(Point(lon, lat)):
            return True
    return False

VERSION_D_CRIDS = ["PGD0", "PID0"]  # if both exist, preferred first: reprocessed, then forward-processed

def keep_best_version_d(granules):
    """Keep one Version D granule per overpass: the preferred CRID, then the highest product counter."""
    best = {}
    for g in granules:
        name = g["umm"]["GranuleUR"].removesuffix("_swot")
        overpass, crid, counter = name.rsplit("_", 2)  # e.g. ..._20250413T095502, PGD0, 01
        if crid not in VERSION_D_CRIDS:
            continue  # skip Version C granules
        rank = (VERSION_D_CRIDS.index(crid), -int(counter))
        if overpass not in best or rank < best[overpass][0]:
            best[overpass] = (rank, g)
    return sorted((g for rank, g in best.values()), key=lambda g: g["umm"]["GranuleUR"])

covering = [g for g in louisville_raster if footprint_covers(g, site_lon, site_lat)]
louisville_raster = keep_best_version_d(covering)
for g in louisville_raster:
    print(g["umm"]["GranuleUR"], round(g.size, 1), "MB")

You should see 9 granules, all `UTM16S`, each about 60–80 MB.

Read a Raster name from left to right: resolution (`100m`), UTM zone and latitude band (`UTM16S`), **cycle** (`030`, one 21-day repeat of the orbit), **pass** (`175`, one numbered track within the cycle), scene number (`111F`), start and end time (UTC), CRID and product counter. Four passes see Louisville (`160`, `175`, `466`, `481`), so the point is observed about four times in each 21-day cycle, in pairs half a day apart.

### Find RiverSP granules over the Ohio at Louisville

RiverSP granule metadata only has a rough rectangle for the whole continental pass, so `footprint_covers` can't screen them. Instead, keep the RiverSP granules from the same **cycle and pass** as the Raster scenes that cover the point.

In [ ]:
louisville_reach_granules = earthaccess.search_data(
    short_name="SWOT_L2_HR_RiverSP_D",
    granule_name="*Reach*",          # reach files only
    bounding_box=site_bbox,
    temporal=(start, end),
)
# Raster name split on "_": [5] UTM zone, [10] cycle, [11] pass, [12] scene. RiverSP: [5] cycle, [6] pass.
# (cycle, pass) of each Raster scene that covers the point, e.g. ("031", "175")
seen_passes = {tuple(g["umm"]["GranuleUR"].split("_")[10:12]) for g in louisville_raster}
louisville_reach_granules = keep_best_version_d(
    [g for g in louisville_reach_granules if tuple(g["umm"]["GranuleUR"].split("_")[5:7]) in seen_passes]
)
print(len(louisville_reach_granules), "reach granules from passes that see the point")
for g in louisville_reach_granules[:3]:
    print(g["umm"]["GranuleUR"], round(g.size, 1), "MB")

You should see 9 reach granules from passes that see the point, about 8 MB each. RiverSP names carry the feature type (`Reach`), cycle, pass, continent (`NA`, North America), start and end time, CRID and product counter.

**Back to the slides.**

**Live coding 2: download SWOT data.** Run from here to the end of *Understanding what you downloaded*, then go back to the slides. If your kernel restarted, use **Run All Above** from this cell first.

## Programmatic data downloads

Keep downloads exactly as they arrived in `data_dir`, and write anything you compute from them somewhere else. Then you can always tell raw data from derived data, and re-create the derived files from the raw ones.

### RiverSP reaches with `earthaccess`

RiverSP granules are zipped shapefiles, so you read them with `geopandas`. Download the granule from 13 April 2025 (cycle 031, pass 175), close to the time of the flood crest, and find the SWORD reach closest to the example point. This is also how you find a `reach_id` to use with `hydrocron` if you don't have one. (You can also look up reach IDs interactively in [SWORD Explorer](https://www.swordexplorer.com/).)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

flood_granule = [g for g in louisville_reach_granules if "_031_175_" in g["umm"]["GranuleUR"]]
files = earthaccess.download(flood_granule, local_path=data_dir)
reaches = gpd.read_file(files[0])
print(len(reaches), "reaches in this granule")

# Distance from each reach to the point, in meters, in the UTM zone around the point (here zone 16N)
utm_crs = gpd.GeoSeries([Point(site_lon, site_lat)], crs="EPSG:4326").estimate_utm_crs()
site = gpd.GeoSeries([Point(site_lon, site_lat)], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
reaches_utm = reaches.to_crs(utm_crs)
reaches_utm["dist_to_site_m"] = reaches_utm.distance(site).round()
reaches_utm.nsmallest(3, "dist_to_site_m")[["reach_id", "river_name", "dist_to_site_m", "p_width", "wse", "width", "reach_q", "time_str"]]

You should see 1153 reaches in this granule. The point sits on reach **`74267300251`**, about 200 m from its centerline. Its neighbors are `74267300241` (downstream) and `74267300261` (upstream). RiverSP files carry about 130 columns. Each row is one reach seen on this overpass. The key columns:

- `reach_id` is the **Location identifier**.
- `wse` (water surface elevation) and `width` are the main **Variables**. Their **Variable unit** is meters; `wse` is height above the EGM2008 geoid (a model of mean sea level).
- `wse_u` and `width_u` are each value's estimated uncertainty, in meters.
- `p_width` is SWORD's prior (expected) width for the reach, useful for judging whether a measured `width` is plausible. SWORD expects this reach to be about 720 m wide, and SWOT measured 853 m on this overpass, with the river in flood.
- `width` is derived from area: `area_total`, the water area assigned to the reach, divided by `p_length`, SWORD's length for the reach. That makes it an average width over the reach, not a bank-to-bank measurement. Raster has no width variable: it reports `water_area` and `water_frac` for each grid cell.
- `reach_q` is the summary **Data quality flag**: 0 = good, 1 = suspect, 2 = degraded, 3 = bad.
- `time_str` is the overpass **Time** in UTC.
- Missing values are stored as `-999999999999`, not `NaN`. Filter them out before plotting.

This cell draws a location map from this one file:

In [ ]:
import matplotlib.pyplot as plt

near = reaches_utm.nsmallest(3, "dist_to_site_m").to_crs(4326)
fig, (ax_pass, ax_zoom) = plt.subplots(1, 2, figsize=(11, 5.5), width_ratios=[1, 1.4])
# Left: every reach in the granule shows the extent of one pass across North America
reaches.plot(ax=ax_pass, color="0.55", linewidth=0.6)
ax_pass.plot(site_lon, site_lat, marker="*", color="#e34948", markersize=14, markeredgecolor="white")
ax_pass.set_title("All reaches in one RiverSP granule (pass 175)")
# Right: the three reaches nearest the point, labeled with their IDs
for (_, reach), color in zip(near.iterrows(), ["#2a78d6", "#eb6834", "#1baf7a"]):
    gpd.GeoSeries([reach.geometry]).plot(ax=ax_zoom, color=color, linewidth=3)
    x, y = reach.geometry.interpolate(0.5, normalized=True).coords[0]
    ax_zoom.annotate(str(reach.reach_id), (x, y), xytext=(0, 12), textcoords="offset points", ha="center", fontsize=9)

ax_zoom.plot(site_lon, site_lat, marker="*", color="#e34948", markersize=16, markeredgecolor="white")
ax_zoom.set_title(f"Nearest SWORD reaches to the point: {site_name}")
for ax in (ax_pass, ax_zoom):
    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")

fig.tight_layout()
plt.show()

On the left is every reach in one granule, one pass across North America. On the right are the three reaches nearest the point, labeled with their IDs.

### Raster water area with `earthaccess`

Raster granules are NetCDF files, so you read them with `xarray`. Each file is a fixed scene, about 150 km on a side, on a 100 m grid in UTM coordinates. Comparisons only make sense between scenes seen from the **same pass**, because each pass views the river from a different angle and covers a different part of the scene. Download the three pass-175 scenes: before the flood, near the crest, and after.

In [ ]:
import xarray as xr

pass_175 = [g for g in louisville_raster if g["umm"]["GranuleUR"].split("_")[11] == "175"]
raster_files = sorted(earthaccess.download(pass_175, local_path=data_dir))  # in time order
ds = xr.open_dataset(raster_files[0])
ds[["water_area", "water_frac", "wse", "water_area_qual"]]

You should see a dataset of 1511 × 1511 pixels. For each 100 m pixel:

- `water_area` (**Variable unit**: m²) is the surface area of water in the pixel. A pixel that is fully water is about 10,000 m².
- `water_frac` (unitless) is the fraction of the pixel covered by water.
- `wse` (m above the geoid) is the water surface elevation.
- `water_area_qual` is the **Data quality flag** for `water_area`: 0 = good, 1 = suspect, 2 = degraded, 3 = bad. `wse_qual` and the other `_qual` variables work the same way.

Pixels outside the swath (the strip of ground the radar imaged on this overpass) are `NaN`. The `x` and `y` coordinates are UTM meters, and the full projection is in the `crs` variable's `crs_wkt` attribute.

To compare overpasses, add up `water_area` in a 10 km × 10 km box around the point in each file. The function uses each file's own projection, so the box stays in the same place. It counts the pixels at each flag level and reports the area two ways: good and suspect pixels only (`water_area_qual <= 1`), and everything except bad pixels (`<= 2`).

In [ ]:
import pandas as pd
import pyproj

def water_area_near(path, lon=site_lon, lat=site_lat, half_width_m=5000):
    """Water area (km^2) and pixel counts by quality flag in a square box centred on (lon, lat)."""
    ds = xr.open_dataset(path)
    utm = pyproj.CRS.from_wkt(ds["crs"].attrs["crs_wkt"])
    x, y = pyproj.Transformer.from_crs("EPSG:4326", utm, always_xy=True).transform(lon, lat)
    window = ds.sel(x=slice(x - half_width_m, x + half_width_m), y=slice(y - half_width_m, y + half_width_m))
    area = window["water_area"]
    qual = window["water_area_qual"].where(area.notnull())  # only pixels SWOT observed
    return {
        "time": ds.attrs["time_granule_start"][:16],
        "px_good_suspect": int((qual <= 1).sum()),
        "px_degraded": int((qual == 2).sum()),
        "px_bad": int((qual == 3).sum()),
        "area_good_suspect_km2": round(float(area.where(qual <= 1).sum()) / 1e6, 1),
        "area_not_bad_km2": round(float(area.where(qual <= 2).sum()) / 1e6, 1),
    }

water_area = pd.DataFrame([water_area_near(f) for f in raster_files])
water_area

You should see three rows. The `area_good_suspect_km2` column reads 14.2, 0.0 and 15.2, and `area_not_bad_km2` reads 14.2, 16.4 and 15.2.

The 10 km box holds 10,000 pixels, and about three quarters were observed on each overpass. Two things stand out:

1. **A strict filter would hide the flood.** On 13 April, near the crest, *every* observed pixel is flagged degraded (2), so `area_good_suspect_km2` is 0.0. That doesn't mean there was no water.
2. **Most of the flood was vertical.** The water area grew by about 15%, while the water surface rose by more than 8 m (you'll see that in the `hydrocron` time series below). Water area in a fixed box is a blunt measure on a large river whose banks hold most of the rise.

*Optional:* the flag says *that* the 13 April pixels are degraded. The bitwise quality flag, `water_area_qual_bitwise`, says *why*: each bit is one reason, and the variable's attributes name them. `values & mask` (bitwise AND) is non-zero when that reason's bit is set. This cell counts, across the whole 13 April scene, how many observed pixels have each "degraded" reason set.

In [ ]:
import numpy as np

ds = xr.open_dataset(raster_files[1])   # the 13 April scene
bitwise = ds["water_area_qual_bitwise"].where(ds["water_area"].notnull())   # observed pixels only
values = bitwise.values[np.isfinite(bitwise.values)].astype("int64")
reasons = dict(zip(bitwise.attrs["flag_meanings"].split(), bitwise.attrs["flag_masks"]))
print(len(values), "observed pixels in the scene")
{reason: int(((values & int(mask)) > 0).sum()) for reason, mask in reasons.items() if "degraded" in reason}

You should see 684605 observed pixels, all with `geolocation_qual_degraded`.

The whole overpass was flagged because its geolocation (where each pixel sits on the ground) is less certain, not because of anything about the river. The heights are likely not trustworthy, but the water area might be reasonable. Whether to use such a scene is your call. Here, the river in the 13 April scene lines up with the river in the other two (see the maps below), so this lesson keeps it and says so.

Save derived results like this table outside the raw-data folder, with a name that says what they are.

In [ ]:
import os
os.makedirs("data/swot/derived", exist_ok=True)
water_area.to_csv("data/swot/derived/louisville_water_area_pass175.csv", index=False)

Maps show the change more clearly than totals. This cell plots `water_frac` in a 30 km window for each of the three scenes, leaving out only pixels flagged bad, so the degraded 13 April scene still appears.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharey=True)
for ax, path in zip(axes, raster_files):
    ds = xr.open_dataset(path)
    utm = pyproj.CRS.from_wkt(ds["crs"].attrs["crs_wkt"])
    x, y = pyproj.Transformer.from_crs("EPSG:4326", utm, always_xy=True).transform(site_lon, site_lat)
    window = ds.sel(x=slice(x - 15000, x + 15000), y=slice(y - 15000, y + 15000))
    frac = window["water_frac"].where(window["water_area_qual"] <= 2)  # drop pixels flagged bad
    image = ax.pcolormesh((window.x - x) / 1000, (window.y - y) / 1000, frac.clip(0, 1), cmap="Blues", vmin=0, vmax=1)
    ax.plot(0, 0, marker="*", color="#e34948", markersize=12, markeredgecolor="white")
    ax.set_title(ds.attrs["time_granule_start"][:10])
    ax.set_xlabel("km east of the point")
    ax.set_aspect("equal")

axes[0].set_ylabel("km north of the point")
fig.colorbar(image, ax=axes, label="Water fraction (pixels not flagged bad)", shrink=0.85)
plt.show()

You should see the Ohio River as a dark blue curving band in all three maps. On 13 April the band is visibly wider, especially downstream to the southwest, and a few side channels and low areas next to the river are filled.

### RiverSP time series with `hydrocron`

If what you want is a time series for one reach, `hydrocron` is the tool for the job. It's a web API, so there is nothing extra to install: any HTTP client works, and this lesson uses `requests`. You don't need an Earthdata login or an API key for normal use. One request returns one feature (a reach, node or lake) over a time range, and responses are capped at 6 MB. See the [`hydrocron` timeseries documentation](https://podaac.github.io/hydrocron/timeseries) for every parameter.

The function below, adapted from the CUAHSI longitudinal-profile notebook, wraps one request and returns a `pandas` DataFrame. It drops overpasses with no valid measurement (fill values) and, by default, keeps everything else. Pass `max_reach_q` to also drop observations whose quality flag is worse than you can accept.

In [ ]:
import io
import pandas as pd
import requests

HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
FILL_VALUE = -999999999999.0
FIELDS = "reach_id,time_str,wse,wse_u,width,p_width,reach_q,crid"

def get_reach_timeseries(reach_id, start, end, fields=FIELDS, max_reach_q=3):
    """One hydrocron request: a DataFrame of valid RiverSP observations of one reach between start and end (UTC)."""
    params = {
        "feature": "Reach",
        "feature_id": reach_id,
        "start_time": start,
        "end_time": end,
        "output": "csv",
        "fields": fields,
    }
    response = requests.get(HYDROCRON_URL, params=params, timeout=120)
    if response.status_code != 200:
        raise RuntimeError(f"hydrocron returned HTTP {response.status_code}: {response.text[:300]}")
    df = pd.read_csv(io.StringIO(response.json()["results"]["csv"]))
    df = df[(df["wse"] != FILL_VALUE) & (df["reach_q"] <= max_reach_q)].copy()
    df["time"] = pd.to_datetime(df["time_str"])
    return df

louisville = get_reach_timeseries(reach_id, start, end)   # reach_id, start and end come from the query block
print(len(louisville), "observations; reach_q counts:", louisville["reach_q"].value_counts().sort_index().to_dict())
louisville[["time_str", "wse", "wse_u", "width", "reach_q", "crid"]]

You should see 12 observations, 10 flagged 1 and 2 flagged 2. Every row is one SWOT overpass of the reach. `wse`, `wse_u` and `width` are in meters; `hydrocron` also adds a `<field>_units` column (for example `wse_units`) giving each **Variable unit**, and the function adds a `time` column parsed as a timestamp for plotting. `crid` records which processing produced each value (here, all reprocessed Version D, `PGD0`). `hydrocron` also returns a row for each overpass that produced no valid measurement, with `time_str` = `no_data`, `wse` = `-999999999999` and `reach_q` = 3; the function drops those.

A plot shows the flood clearly.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for q, color, label in [(1, "#2a78d6", "suspect (reach_q = 1)"), (2, "#eb6834", "degraded (reach_q = 2)")]:
    rows = louisville[louisville["reach_q"] == q]
    ax.errorbar(rows["time"], rows["wse"], yerr=rows["wse_u"], fmt="o", color=color, markersize=7, capsize=3, label=label)

ax.plot(louisville["time"], louisville["wse"], color="0.7", linewidth=1, zorder=0)
ax.set_ylabel("WSE (m above the geoid)")
ax.set_title(f"SWOT reach {reach_id}, {site_name}")
ax.legend(title="Quality flag", loc="upper right")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

You should see values between about 121 and 123.5 m until early April, a jump to about 131 m on 12 and 13 April, and a drop back to about 121 m by 23 April. Ten points are suspect and two are degraded (22 March and 13 April). The error bars show `wse_u`, about ±0.1 m, smaller than the markers.

SWOT saw this reach 12 times in two months, in pairs about 13 hours apart (passes 160 and 175, then 466 and 481).

- **The flood is obvious.** The water surface sat around 121–123 m in March, jumped to 131.2 m on 12 April, and was back near 121 m by 23 April. The two overpasses in most pairs agree to within 0.8 m; the biggest gap (1.7 m, 22–23 March) involves a degraded value.
- **No observation is flagged good.** Ten are suspect (`reach_q` = 1) and two degraded (2). That is normal for Version D RiverSP. A `reach_q == 0` filter (`get_reach_timeseries(..., max_reach_q=0)`) would return an empty table.
- **Width is a useful sanity check.** The measured `width` stays between about 720 and 960 m, close to SWORD's `p_width` of 720 m and widest at the crest. A measured width far from `p_width` is a warning sign that SWOT saw only part of the channel or counted water outside it, and on narrower rivers it is often the first clue that a `wse` value is off.

If a reach ID does not exist in the collection, `hydrocron` answers with an HTTP 400 and a message such as `Results with the specified Feature ID ... were not found`. The function raises that as an error, so a typo doesn't pass silently.

## Understanding what you downloaded

Every data product in this course is described with the same shared concepts. Here they are for the three things you retrieved:

| Shared concept | RiverSP reach file (`earthaccess`) | Raster scene (`earthaccess`) | `hydrocron` time series |
|---|---|---|---|
| Location identifier | `reach_id` (SWORD), e.g. `74267300251` | none per pixel: UTM `x`, `y` coordinates on a fixed scene grid | the `reach_id` you asked for |
| Variable | `wse`, `width` (plus slope, area and ~130 other columns) | `water_area`, `water_frac`, `wse` | the `fields` you asked for |
| Variable unit | meters (`wse` above the EGM2008 geoid) | m² (`water_area`), unitless (`water_frac`), m (`wse`) | `<field>_units` columns |
| Time | `time_str`, the overpass time (UTC), one value per reach | `time_granule_start` attribute, one per scene | `time_str`, one row per overpass |
| Data quality flag(s) | `reach_q` (0–3) and its bitwise version `reach_q_b` | `water_area_qual`, `wse_qual` (0–3) | whichever flags you request (`reach_q`) |
| Version / provenance | Version D; CRID in the file name (`PGD0`); collection DOI [10.5067/SWOT-RIVERSP-D](https://doi.org/10.5067/SWOT-RIVERSP-D) | Version D; CRID in the file name; collection DOI [10.5067/SWOT-RASTER-D](https://doi.org/10.5067/SWOT-RASTER-D) | `crid` field; collection name (default Version D) |
| Data unit | one granule = one pass over one continent, all reaches | one granule = one ~150 km scene from one pass | one response = one reach over your time range |

- **No discharge here yet.** SWOT Level 2 discharge estimates are not yet ready for research use as of Nov 2026. Level 2 RiverSP files have discharge columns (names starting `dschg_`), but expect fill values in those columns, and check the current release notes before relying on them.
- **Uncertainties are estimates.** Each RiverSP measurement has a matching uncertainty column ending in `_u`. For what they mean, see the Product Description Document on the [RiverSP collection page](https://doi.org/10.5067/SWOT-RIVERSP-D), under Docs & Resources > User Guides.
- **Versions change values.** A reprocessed version can change the numbers for the same overpass. Record the collection, the CRID and your access date, and cite the collection DOI.

**Back to the slides.**

## Now you try it

**Now you try it: about 12 minutes.** Work through the steps below, then go back to the slides. A good pace is to reach step 4 by about minute 6.

Repeat the main steps for accessing SWOT hydrologic data on a second river: the **Willamette River at Salem, Oregon**.

**Your task:** Find the SWORD reach for the Willamette, retrieve its water surface elevation time series with `hydrocron`, then check it against its flags and decide which values you trust.

Make your changes in the code cells below. They use new names (for example `salem_raster`), so your Louisville results stay as they are. Fill in each `...`, using the code above. If a cell stops with `cannot unpack` or a similar error about `ellipsis`, there's still a `...` to fill in. An `IndexError` means an empty list: check you replaced every `site_*` and `seen_passes`.

1. Run the new query for Salem: it's filled in for you in the cell below.
2. Search the Raster collection for the new box and dates, then screen the results with `footprint_covers` and `keep_best_version_d`, as you did for Louisville. Salem is in UTM zone **10**, so the real matches are `UTM10T` scenes. Collect their `(cycle, pass)` pairs in `salem_passes`.
3. Search the RiverSP collection (`granule_name="*Reach*"`), keep the granules from those passes, download **one** of them, and find the reach nearest the point. Any one of them will do: every pass you kept sees the point. `estimate_utm_crs()` picks the right UTM zone for Salem by itself.
4. Set `salem_reach_id` to that reach and call `get_reach_timeseries(salem_reach_id, salem_start, salem_end)`.
5. Plot `wse` over time colored by `reach_q`. Compare `width` with `p_width`, and each value with its neighbors.
6. Download the Raster scenes from 23 and 26 February, a degraded overpass and the overpass after it, and map `water_frac` in a 10 km box around the point.
7. Decide whether to keep the three degraded values, and write down why.

**Short on time or downloads running long?** Skip step 6 and use the numbers in its "What you should get".

### Steps 1 to 3: find the reach

In [ ]:
# Step 1: a new query for Salem (new names, so the Louisville variables are kept)
salem_name = "Willamette River at Salem, OR"
salem_lon, salem_lat = -123.043, 44.944
salem_bbox = (salem_lon - 0.02, salem_lat - 0.02, salem_lon + 0.02, salem_lat + 0.02)
salem_start, salem_end = "2026-02-01T00:00:00Z", "2026-03-31T23:59:59Z"

# Step 2: Raster scenes that really cover the point (see "Find Raster granules over the Ohio at Louisville")
salem_raster = ...           # TODO: search the Raster collection with salem_bbox and your dates
salem_raster = ...           # TODO: screen them with footprint_covers and keep_best_version_d
salem_passes = ...           # TODO: the (cycle, pass) of each scene, as for seen_passes

# Step 3: RiverSP reach granules from those passes (see "Find RiverSP granules over the Ohio at Louisville")
salem_reach_granules = ...   # TODO: search RiverSP (reach files only) with salem_bbox and your dates
salem_reach_granules = ...   # TODO: keep the granules from salem_passes, with keep_best_version_d
salem_files = earthaccess.download(salem_reach_granules[:1], local_path=data_dir)
salem_utm = gpd.GeoSeries([Point(salem_lon, salem_lat)], crs="EPSG:4326").estimate_utm_crs()   # zone 10N here
salem_reaches = gpd.read_file(salem_files[0]).to_crs(salem_utm)
salem_site = gpd.GeoSeries([Point(salem_lon, salem_lat)], crs="EPSG:4326").to_crs(salem_utm).iloc[0]
salem_reaches["dist_to_site_m"] = salem_reaches.distance(salem_site).round()
print(salem_reaches.nsmallest(1, "dist_to_site_m")[["reach_id", "river_name", "dist_to_site_m", "p_width"]])

<details>
<summary><b>Show the example answer</b></summary>

```python
# Step 1: a new query for Salem (new names, so the Louisville variables are kept)
salem_name = "Willamette River at Salem, OR"
salem_lon, salem_lat = -123.043, 44.944
salem_bbox = (salem_lon - 0.02, salem_lat - 0.02, salem_lon + 0.02, salem_lat + 0.02)
salem_start, salem_end = "2026-02-01T00:00:00Z", "2026-03-31T23:59:59Z"

# Step 2: Raster scenes that really cover the point
salem_raster = earthaccess.search_data(short_name="SWOT_L2_HR_Raster_100m_D", bounding_box=salem_bbox, temporal=(salem_start, salem_end))
salem_raster = keep_best_version_d([g for g in salem_raster if footprint_covers(g, salem_lon, salem_lat)])
salem_passes = {tuple(g["umm"]["GranuleUR"].split("_")[10:12]) for g in salem_raster}

# Step 3: RiverSP reach granules from those passes; download one and find the nearest reach
salem_reach_granules = earthaccess.search_data(short_name="SWOT_L2_HR_RiverSP_D", granule_name="*Reach*", bounding_box=salem_bbox, temporal=(salem_start, salem_end))
salem_reach_granules = keep_best_version_d([g for g in salem_reach_granules if tuple(g["umm"]["GranuleUR"].split("_")[5:7]) in salem_passes])
salem_files = earthaccess.download(salem_reach_granules[:1], local_path=data_dir)
salem_utm = gpd.GeoSeries([Point(salem_lon, salem_lat)], crs="EPSG:4326").estimate_utm_crs()   # zone 10N here
salem_reaches = gpd.read_file(salem_files[0]).to_crs(salem_utm)
salem_site = gpd.GeoSeries([Point(salem_lon, salem_lat)], crs="EPSG:4326").to_crs(salem_utm).iloc[0]
salem_reaches["dist_to_site_m"] = salem_reaches.distance(salem_site).round()
print(salem_reaches.nsmallest(1, "dist_to_site_m")[["reach_id", "river_name", "dist_to_site_m", "p_width"]])
```

</details>

**What you should get:** the nearest reach is `78220000131`, about 40 m from the point, with `p_width` = 150 m.

### Step 4: get the series

In [ ]:
# Step 4: the time series
salem_reach_id = ...         # TODO: the reach printed by the cell above, as a string
salem = ...                  # TODO: call get_reach_timeseries with your reach and dates
print(len(salem), "observations; reach_q counts:", salem["reach_q"].value_counts().sort_index().to_dict())
salem[["time_str", "wse", "wse_u", "width", "reach_q"]]

<details>
<summary><b>Show the example answer</b></summary>

```python
# Step 4: the time series
salem_reach_id = "78220000131"   # the nearest reach printed above
salem = get_reach_timeseries(salem_reach_id, salem_start, salem_end)
print(len(salem), "observations; reach_q counts:", salem["reach_q"].value_counts().sort_index().to_dict())
salem[["time_str", "wse", "wse_u", "width", "reach_q"]]
```

</details>

**What you should get:** `hydrocron` returns 9 observations from 3 February to 29 March 2026: 6 suspect (`reach_q` = 1) and 3 degraded (`reach_q` = 2).

### Step 5: plot and compare

In [ ]:
# Step 5: wse colored by reach_q, and width against SWORD's p_width
fig, (ax_wse, ax_width) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for q, color, label in [(1, "#2a78d6", "suspect (reach_q = 1)"), (2, "#eb6834", "degraded (reach_q = 2)")]:
    rows = ...               # TODO: the rows of salem with this flag
    ax_wse.scatter(rows["time"], rows["wse"], color=color, label=label)
    ax_width.scatter(rows["time"], rows["width"], color=color)
ax_width.axhline(..., color="0.4", linestyle="--", label="SWORD p_width")   # TODO: the reach's p_width
ax_wse.set_ylabel("WSE (m above the geoid)")
ax_width.set_ylabel("Width (m)")
ax_wse.legend()
ax_width.legend()
fig.autofmt_xdate()
plt.show()

<details>
<summary><b>Show the example answer</b></summary>

```python
# Step 5: wse colored by reach_q, and width against SWORD's p_width
fig, (ax_wse, ax_width) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for q, color, label in [(1, "#2a78d6", "suspect (reach_q = 1)"), (2, "#eb6834", "degraded (reach_q = 2)")]:
    rows = salem[salem["reach_q"] == q]
    ax_wse.scatter(rows["time"], rows["wse"], color=color, label=label)
    ax_width.scatter(rows["time"], rows["width"], color=color)
ax_width.axhline(salem["p_width"].iloc[0], color="0.4", linestyle="--", label="SWORD p_width")
ax_wse.set_ylabel("WSE (m above the geoid)")
ax_width.set_ylabel("Width (m)")
ax_wse.legend()
ax_width.legend()
fig.autofmt_xdate()
plt.show()
```

</details>

**What you should get:** the suspect values lie between about 32.7 and 35.9 m. The three degraded ones (3 and 23 February, 16 March) are 37–38 m, several meters higher, and their widths (196–253 m) are well above `p_width`. All three come from the same pass, 274.

### Step 6: map it

In [ ]:
# Step 6: map water_frac for a degraded overpass (23 February) and the next overpass (26 February)
first_date, next_date = ...  # TODO: the two dates, written as in a granule name, such as "20260223T"
half_width_m = ...           # TODO: half of the box width, in meters
compare = [g for g in salem_raster if first_date in g["umm"]["GranuleUR"] or next_date in g["umm"]["GranuleUR"]]
compare_files = sorted(earthaccess.download(compare, local_path=data_dir))   # in time order
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)
for ax, path in zip(axes, compare_files):
    ds = xr.open_dataset(path)
    utm = pyproj.CRS.from_wkt(ds["crs"].attrs["crs_wkt"])
    x, y = pyproj.Transformer.from_crs("EPSG:4326", utm, always_xy=True).transform(salem_lon, salem_lat)
    window = ds.sel(x=slice(x - half_width_m, x + half_width_m), y=slice(y - half_width_m, y + half_width_m))
    keep = window["water_area_qual"] <= 2                                    # drop pixels flagged bad
    area_km2 = float(window["water_area"].where(keep).sum()) / 1e6
    print(ds.attrs["time_granule_start"][:10], round(area_km2, 1), "km² of water in the 10 km box")
    ax.pcolormesh((window.x - x) / 1000, (window.y - y) / 1000, window["water_frac"].where(keep).clip(0, 1), cmap="Blues", vmin=0, vmax=1)
    ax.plot(0, 0, marker="*", color="#e34948", markersize=12, markeredgecolor="white")
    ax.set_title(ds.attrs["time_granule_start"][:10])
    ax.set_xlabel("km east of the point")
    ax.set_aspect("equal")
axes[0].set_ylabel("km north of the point")
plt.show()

<details>
<summary><b>Show the example answer</b></summary>

```python
# Step 6: map water_frac for a degraded overpass (23 February) and the next overpass (26 February)
compare = [g for g in salem_raster if "20260223T" in g["umm"]["GranuleUR"] or "20260226T" in g["umm"]["GranuleUR"]]
compare_files = sorted(earthaccess.download(compare, local_path=data_dir))   # in time order
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)
for ax, path in zip(axes, compare_files):
    ds = xr.open_dataset(path)
    utm = pyproj.CRS.from_wkt(ds["crs"].attrs["crs_wkt"])
    x, y = pyproj.Transformer.from_crs("EPSG:4326", utm, always_xy=True).transform(salem_lon, salem_lat)
    window = ds.sel(x=slice(x - 5000, x + 5000), y=slice(y - 5000, y + 5000))
    keep = window["water_area_qual"] <= 2                                    # drop pixels flagged bad
    area_km2 = float(window["water_area"].where(keep).sum()) / 1e6
    print(ds.attrs["time_granule_start"][:10], round(area_km2, 1), "km² of water in the 10 km box")
    ax.pcolormesh((window.x - x) / 1000, (window.y - y) / 1000, window["water_frac"].where(keep).clip(0, 1), cmap="Blues", vmin=0, vmax=1)
    ax.plot(0, 0, marker="*", color="#e34948", markersize=12, markeredgecolor="white")
    ax.set_title(ds.attrs["time_granule_start"][:10])
    ax.set_xlabel("km east of the point")
    ax.set_aspect("equal")
axes[0].set_ylabel("km north of the point")
plt.show()
```

</details>

**What you should get:** on 23 February the box holds about 66 km² of water, spread across the landscape, and on 26 February about 11 km², with the river clear. So on that degraded overpass SWOT seems to have counted water outside the channel, which is why the width is so far above `p_width`. Those Raster pixels are mostly flagged suspect, so the Raster flag alone wouldn't show you this.

These two scenes are from different passes, which you wouldn't normally compare. All three degraded values are pass 274, so no clean scene from that pass falls in this window. Treat the maps as a look at the pattern, not a measured change.

### Step 7: decide

Which values do you keep, and why? Write your decision in the cell below.

*Your decision:*

<details>
<summary><b>Show a reasonable call</b></summary>

A reasonable call is to leave the three degraded values out and write down why. An independent record for those dates, such as a stream gage or imagery, would tell you more.

Notice that the same flag led to different decisions: this lesson keeps the degraded 13 April value at Louisville and leaves these out, because the evidence differed.

</details>

**Back to the slides.**

## Further reading

- [`earthaccess` documentation](https://earthaccess.readthedocs.io/en/latest/), including the [authentication how-to](https://earthaccess.readthedocs.io/en/latest/user/howto/authenticate/) and the [API reference](https://earthaccess.readthedocs.io/en/latest/api/).
- [`hydrocron` documentation](https://podaac.github.io/hydrocron/) and the [timeseries endpoint reference](https://podaac.github.io/hydrocron/timeseries).
- [PO.DAAC Cookbook: SWOT tutorials](https://podaac.github.io/tutorials/quarto_text/SWOT.html), including [Hydrocron API: Getting Started with SWOT Time Series](https://podaac.github.io/tutorials/notebooks/datasets/Hydrocron_SWOT_timeseries_examples_basic.html) and [Hydrocron API: Parallel Time Series Processing with Dask](https://podaac.github.io/tutorials/notebooks/datasets/Hydrocron_SWOT_timeseries_examples.html#set-up-dask-workers-for-parallelization), by Nikki Tebaldi, Cassandra Nickles and Brandi Downs.
- Dataset DOIs: [SWOT Level 2 River Single-Pass Vector, Version D](https://doi.org/10.5067/SWOT-RIVERSP-D) and [SWOT Level 2 Water Mask Raster, Version D](https://doi.org/10.5067/SWOT-RASTER-D).
- [Earthdata Search](https://search.earthdata.nasa.gov/), [SWOTViz](https://swotviz.cuahsi.io/), [WaterMAP](https://apps.usgs.gov/watermap/#/?lat=48.87204&lon=-125.953207&z=2.5&data=swotRiverReaches) (USGS) and [SWORD Explorer](https://www.swordexplorer.com/).

## About this notebook

Released under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.html) (GPL-3.0). You may use, change and share it under the same license.

Adapted from [SWOT - River Longitudinal Profiles for Water Resources](https://www.hydroshare.org/resource/1e5dff3ab72440d2a4e3942326ada5a4/) by Mike Durand, with contributions from Bidhya Yadav (Ohio State University), CUAHSI notebooks (GPL-3.0). The `get_reach_timeseries` function adapts `PullReachTimeseries` from this source.

**How to cite the course:** Platt, L., Wong, K., Vaarre-Lamoureux, K., & Bogan, A. *Reproducible Workflows with Open Water Data: NASA, NOAA, USGS River Data Access* (Version pilot-2026-11) [Learning material (online course)]. CUAHSI. GPL-3.0.

**How to cite the data:** cite the collection DOI with your access date, for example: *SWOT. (2025). SWOT Level 2 River Single-Pass Vector Data Product [Dataset]. NASA Physical Oceanography Distributed Active Archive Center. https://doi.org/10.5067/SWOT-RIVERSP-D. Accessed 2026-10-08.*

**Acknowledgements:** this course was developed by CUAHSI with contributions from colleagues at NOAA, USGS and NASA. This material is based upon work supported by the U.S. National Science Foundation under [Award No. 2535162](https://www.nsf.gov/awardsearch/show-award?AWD_ID=2535162). Any opinions, findings, and conclusions or recommendations expressed in this material are those of the authors and do not necessarily reflect the views of the National Science Foundation. Generative AI (Anthropic's Claude) was used to help draft content, and contributors and staff reviewed, edited and tested all of it.